<a href="https://colab.research.google.com/github/Anaaaslagi/Tugas1A_AnasGhifari_5026221155/blob/main/Week2_PBA_Scrapping_(Anas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Scrapping Transjakarta Mobile

In [1]:
!pip install google_play_scraper
!pip install textblob
!pip install seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00


In [2]:
from google_play_scraper import app
import pandas as pd
import numpy as np
import sklearn
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
import textblob
#from wordcloud import WordCloud
from pathlib import Path
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report, accuracy_score

import pickle
import re
import time
import datetime                              # access to %%time, for timing individual notebook cells
import os
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline
%config InlineBackend.figure_format='retina'

# Import seaborn styles explicitly
import seaborn as sns
# Apply the seaborn style before creating plots
sns.set_style("whitegrid")  # This line sets the Seaborn style

plt.rcParams["figure.figsize"] = (15,10)

In [3]:
#Android App Transjakarta  from Google Play at
#https://play.google.com/store/apps/details?id=com.transjakmobile
#The apps ID found in the link after id=com.transjakmobile


from google_play_scraper import app, Sort, reviews_all

transjakarta_reviews = reviews_all(
    'com.transjakmobile',
    sleep_milliseconds=0, # defaults to 0
    lang='id', # defaults to 'en'
    sort=Sort.NEWEST, # defaults to Sort.MOST_RELEVANT
)

In [4]:
#Save Transjakarta reviews into dataframe
df_tj = pd.DataFrame(np.array(transjakarta_reviews),columns=['content'])
df_tj = df_tj.join(pd.DataFrame(df_tj.pop('content').tolist()))
df_tj.to_csv(r'df_tj', index=False)

In [ ]:
df_tj

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,84e3e653-c75c-4de6-ba9c-40ce51b72a88,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,"Aplikasi Transjakarta benar-benar membantu! Navigasi mudah, tampilannya simpel, dan informasi rute selalu update. Cocok banget buat warga Jabodetabek yang sering naik bus. Fitur pencarian halte da...",5,0,2.8.0,2025-09-23 09:20:02,None,NaT,2.8.0
1,22290dc0-949c-4ecd-9aa0-0eec1f5a442f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,mempermudah untuk mencari driver.. keren,5,0,2.8.0,2025-09-23 08:49:17,None,NaT,2.8.0
2,afa4cde1-4a3b-43b2-bac9-60d42133534a,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,"mempermudah banget buat berpergian, transaksi jadi cepet ga perlu antri. jadwalnya juga jelas.",5,0,None,2025-09-23 07:07:14,None,NaT,None
3,918fc447-b183-4270-b6b7-cac5ec268ec4,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,"aplikasi ini berguna untuk mengetahui halte dan rute, Dan tidak ada iklan yang muncul, bagus",5,0,2.8.0,2025-09-23 06:50:56,None,NaT,2.8.0
4,fdf07e03-387c-410d-8126-9e106a2e8bce,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,"senang sekali ada apk ini, sangat membantu membuat schedule dalam perjalanan dari kantor kerumah tiap, pokoknya aman banget deeh",5,0,2.8.0,2025-09-23 06:41:53,None,NaT,2.8.0
...,...,...,...,...,...,...,...,...,...,...,...
2846,cdd9f5f2-b4cb-44c2-814e-860e0688f3d5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,Mantaps,5,5,1.4,2024-05-16 11:10:02,None,NaT,1.4
2847,66859930-33cc-484a-9c0a-f25534958578,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,Sudah enggak crash kalau pilih profil,5,7,1.8,2024-05-16 04:52:59,None,NaT,1.8
2848,eb386bcd-d074-448e-b636-8f8b4d20ecb1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,"Aplikasi kocag, gk becus bikin aplikasi, masuk kerja dari orang dalem .... Aplikasinya lebih bagus game tetris",1,7,1.4,2024-05-12 09:42:40,None,NaT,1.4
2849,43b40878-dcff-44b8-bed5-2dfb89bc0880,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g,"Aplikasi gak jelas, server eror, loading lama, pencarian rute susah, posisi bus kacau, UI gak menarik, gak jadi niat mau naik tj kalau kaya gini.",1,23,1.4,2024-05-11 13:43:39,"Halo Teguh Aliansyah,\nTerima kasih untuk masukannya dan terima kasih telah menjadi pengguna awal kami. Mohon maaf untuk saat ini kami sedang dalam proses peningkatan layanan aplikasi. Masukan And...",2024-05-23 06:29:46,1.4


In [ ]:
df_tj.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2851 entries, 0 to 2850
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              2851 non-null   object        
 1   userName              2851 non-null   object        
 2   userImage             2851 non-null   object        
 3   content               2851 non-null   object        
 4   score                 2851 non-null   int64         
 5   thumbsUpCount         2851 non-null   int64         
 6   reviewCreatedVersion  2408 non-null   object        
 7   at                    2851 non-null   datetime64[ns]
 8   replyContent          4 non-null      object        
 9   repliedAt             4 non-null      datetime64[ns]
 10  appVersion            2408 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 245.1+ KB
